# E791 single-toy minimization diagnostic

This notebook isolates the failure mode seen in the GenFit bias study.

A **single 50k-event pseudoexperiment** is generated from the E791 Fit-2 truth model. The *same toy* is fitted three times with identical likelihood and Minuit settings, changing only the starting point:

1. exactly at the injected truth;
2. close to the truth;
3. a wide random start.

The main diagnostic is
\[
\Delta\mathrm{NLL}=\mathrm{NLL}_{fit}-\mathrm{NLL}_{truth}.
\]
Since the truth point is inside the fitted parameter space, a successful minimization must satisfy \(\Delta\mathrm{NLL}\le 0\) up to tiny numerical tolerance.

The notebook also scans a common scale applied to all free coefficients while the fixed \(\rho(770)\) coefficient remains \(1+0i\). This tests whether the likelihood develops an asymptotic/runaway direction.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from iminuit import Minuit

from dalitzplotfitter import (
    DecayChannel, DecayModel, NonResonant, Parameter,
    RealImag, Resonance, enable_x64,
)
from dalitzplotfitter.integration import matrix_normalization

enable_x64()


## 1. Build exactly the same E791 Fit-2 model


In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))

fit2_polar = {
    "sigma": (1.17, 205.7),
    "rho770": (1.00, 0.0),
    "NR": (0.48, 57.3),
    "f0_980": (0.43, 165.0),
    "f2_1270": (0.76, 57.3),
    "f0_1370": (0.26, 105.4),
    "rho1450": (0.14, 319.1),
}

def polar_to_xy(r, phase_deg):
    phase = np.deg2rad(phase_deg)
    return r * np.cos(phase), r * np.sin(phase)

def internal_xy(name):
    r, phase = fit2_polar[name]
    if name == "NR":
        phase += 180.0
    return polar_to_xy(r, phase)

truth_xy = {name: internal_xy(name) for name in fit2_polar}

truth = {}

def free_coefficient(name):
    x, y = truth_xy[name]
    truth[f"{name}.x"] = float(x)
    truth[f"{name}.y"] = float(y)
    return RealImag(
        Parameter.coefficient(f"{name}.x", 0.0, owner=name, step=0.01),
        Parameter.coefficient(f"{name}.y", 0.0, owner=name, step=0.01),
    )

coefficients = {
    "sigma": free_coefficient("sigma"),
    "rho770": RealImag(1.0, 0.0),
    "NR": free_coefficient("NR"),
    "f0_980": free_coefficient("f0_980"),
    "f2_1270": free_coefficient("f2_1270"),
    "f0_1370": free_coefficient("f0_1370"),
    "rho1450": free_coefficient("rho1450"),
}

components = [
    Resonance("sigma", (0, 1), coefficients["sigma"], mass=0.4780, width=0.3240, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho770", (0, 1), coefficients["rho770"], mass=0.7693, width=0.1502, spin=1, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_980", (0, 1), coefficients["f0_980"], mass=0.9750, width=0.0440, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f2_1270", (0, 1), coefficients["f2_1270"], mass=1.2750, width=0.1850, spin=2, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_1370", (0, 1), coefficients["f0_1370"], mass=1.4340, width=0.1730, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho1450", (0, 1), coefficients["rho1450"], mass=1.4650, width=0.3100, spin=1, resonance_radius=3.0, parent_radius=3.0),
    NonResonant(coefficients["NR"]),
]

model = DecayModel(channel, components)
free_parameters = tuple(p for p in model.parameters if not p.fixed)
names = tuple(p.name for p in free_parameters)
truth_vector = np.asarray([truth[name] for name in names], dtype=float)

print("number of free parameters:", len(names))
for name, value in zip(names, truth_vector):
    print(f"{name:18s} {value:+.8f}")


## 2. Generate one toy and freeze it

The candidate pool and normalization cache are constructed once. One unweighted toy is sampled from the truth distribution and then reused by **all three fits** below.


In [ ]:
SAMPLE_SIZE = 50_000
POOL_SIZE = 1_000_000
POOL_SEED = 2000
TOY_SEED = 791

normalization_sample = model.normalization_sample
pool = model.generate_phase_space(POOL_SIZE, seed=POOL_SEED)
pool_cache = model.prepare_cache(pool, normalization_sample)

truth_intensity, truth_normalization = pool_cache.evaluate(truth)
target_weights = jnp.asarray(pool.weights * truth_intensity)
probabilities = target_weights / jnp.sum(target_weights)

sample_key = jax.random.key(TOY_SEED)
indices = jax.random.choice(
    sample_key,
    pool.size,
    shape=(SAMPLE_SIZE,),
    replace=True,
    p=probabilities,
)
data_components = pool_cache.data_components[indices]

print("pool size:", pool.size)
print("toy size :", SAMPLE_SIZE)
print("truth normalization:", float(truth_normalization))


## 3. Construct exactly the coefficient-only likelihood used by `GenFit`


In [ ]:
fixed = {
    parameter.name: float(parameter.value)
    for parameter in model.parameters
    if parameter.fixed
}
components_cached = pool_cache.components
normalization_matrix = pool_cache.normalization_matrix_fixed

def vector_nll(vector, data_components):
    values = dict(fixed)
    values.update({name: vector[i] for i, name in enumerate(names)})

    coefficients = jnp.asarray(
        [component.coefficient.value(values) for component in components_cached]
    )
    amplitude = data_components @ coefficients
    intensity = jnp.abs(amplitude) ** 2
    normalization = matrix_normalization(coefficients, normalization_matrix)

    return (
        -jnp.sum(jnp.log(jnp.clip(intensity, min=1e-300)))
        + SAMPLE_SIZE * jnp.log(normalization)
    )

value_and_grad = jax.jit(jax.value_and_grad(vector_nll, argnums=0))

truth_vector_jax = jnp.asarray(truth_vector)
truth_nll, truth_grad = value_and_grad(truth_vector_jax, data_components)

print(f"NLL(truth) = {float(truth_nll):.9f}")
print(f"|grad NLL(truth)| = {np.linalg.norm(np.asarray(truth_grad)):.6e}")


The truth is the **generating parameter**, not necessarily the exact finite-sample MLE, so its gradient is not expected to vanish. But any successful minimizer starting anywhere must be able to reach an NLL no larger than `NLL(truth)`.


## 4. Fit helper and three controlled starting points


In [ ]:
def run_fit(label, start_vector):
    start_vector = np.asarray(start_vector, dtype=float)

    def fcn(*values):
        value, _ = value_and_grad(jnp.asarray(values), data_components)
        return float(value)

    def grad(*values):
        _, gradient = value_and_grad(jnp.asarray(values), data_components)
        return np.asarray(gradient, dtype=float)

    start_nll = fcn(*start_vector)
    fit = Minuit(fcn, *start_vector, name=names, grad=grad)
    fit.errordef = 0.5
    fit.tol = 1e-4
    fit.strategy = 1
    fit.print_level = 0

    for parameter in free_parameters:
        if parameter.bounds is not None:
            fit.limits[parameter.name] = parameter.bounds
        if parameter.step is not None:
            fit.errors[parameter.name] = parameter.step

    fit.migrad(ncall=100_000)
    fit.strategy = 2
    fit.migrad(ncall=100_000)
    fit.hesse()

    fitted = np.asarray([fit.values[name] for name in names], dtype=float)
    errors = np.asarray([fit.errors[name] for name in names], dtype=float)

    return {
        "label": label,
        "fit": fit,
        "start": start_vector,
        "start_nll": float(start_nll),
        "values": fitted,
        "errors": errors,
        "nll": float(fit.fval),
        "delta_nll": float(fit.fval - truth_nll),
        "edm": float(fit.fmin.edm),
        "valid": bool(fit.valid),
        "has_posdef_covar": bool(getattr(fit.fmin, "has_posdef_covar", False)),
        "at_limit": bool(getattr(fit.fmin, "has_parameters_at_limit", False)),
        "distance_to_truth": float(np.linalg.norm(fitted - truth_vector)),
        "max_abs_coefficient": float(np.max(np.abs(fitted))),
    }

rng = np.random.default_rng(12345)

start_truth = truth_vector.copy()
start_near = truth_vector + rng.normal(0.0, 0.05, size=truth_vector.size)
start_random = rng.uniform(-2.5, 2.5, size=truth_vector.size)

results = [
    run_fit("truth start", start_truth),
    run_fit("near truth", start_near),
    run_fit("random", start_random),
]


## 5. Compare the minima


In [ ]:
header = (
    f"{'start':14s} {'valid':>6s} {'start NLL':>14s} {'fit NLL':>14s} "
    f"{'DeltaNLL':>12s} {'EDM':>11s} {'|theta-truth|':>15s} {'max|coef|':>12s}"
)
print(header)
for result in results:
    print(
        f"{result['label']:14s} "
        f"{str(result['valid']):>6s} "
        f"{result['start_nll']:14.5f} "
        f"{result['nll']:14.5f} "
        f"{result['delta_nll']:12.5f} "
        f"{result['edm']:11.3e} "
        f"{result['distance_to_truth']:15.6g} "
        f"{result['max_abs_coefficient']:12.6g}"
    )

print()
print(f"NLL truth = {float(truth_nll):.9f}")


### Interpretation

- If **truth start** and **near truth** converge to similar minima with \(\Delta\mathrm{NLL}<0\), while the random start gives a much larger NLL, the dominant problem is minimization / multiple basins.
- If even **truth start** runs to a pathological solution or cannot beat the truth point, inspect the likelihood implementation and numerical gradients.
- If all three starts converge to the same solution far from truth but with a modest negative \(\Delta\mathrm{NLL}\), that can be a legitimate finite-sample fluctuation. Repeating this for many toys then measures bias.


## 6. Parameter-by-parameter comparison


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(names))
ax.scatter(x, truth_vector, marker="x", s=80, label="truth")

for result in results:
    ax.scatter(x, result["values"], s=35, label=result["label"])

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=70, ha="right")
ax.set_ylabel("coefficient component")
ax.set_title("Same toy: fitted parameters versus injected truth")
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
for result in results:
    print(f"\n=== {result['label']} ===")
    print(f"{'parameter':18s} {'truth':>12s} {'fit':>12s} {'fit-truth':>12s} {'error':>12s}")
    for name, t, v, e in zip(names, truth_vector, result["values"], result["errors"]):
        print(f"{name:18s} {t:12.6f} {v:12.6f} {v-t:12.6f} {e:12.6f}")


## 7. Scan the suspected common-scale direction

All *free* real/imaginary coefficient components are multiplied by a common factor \(\lambda\), while the fixed \(\rho(770)=1+0i\) remains unchanged.

If the NLL keeps decreasing or asymptotically plateaus as \(\lambda\to\) large values, the current reference-amplitude parametrization has a dangerous near-degenerate direction.


In [ ]:
scale_values = np.concatenate([
    np.linspace(0.1, 2.0, 80),
    np.logspace(np.log10(2.05), 4.0, 120),
])

scale_nll = np.asarray([
    float(vector_nll(jnp.asarray(scale * truth_vector), data_components))
    for scale in scale_values
])
delta_scale_nll = scale_nll - float(truth_nll)

best_index = int(np.argmin(scale_nll))
print("best scan scale:", scale_values[best_index])
print("best scan DeltaNLL:", delta_scale_nll[best_index])
print("DeltaNLL at largest scale:", delta_scale_nll[-1])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(scale_values, delta_scale_nll)
ax.axhline(0.0, linestyle="--")
ax.axvline(1.0, linestyle="--", label="truth scale")
ax.set_xscale("log")
ax.set_xlabel("common scale applied to free coefficients")
ax.set_ylabel("NLL(scale) - NLL(truth)")
ax.set_title("Common free-coefficient scale scan")
ax.legend()
fig.tight_layout()
plt.show()


## 8. NLL along the line from truth to each fitted solution

This distinguishes a smoothly connected better minimum from a fit that ended in a disconnected/local basin.


In [ ]:
alphas = np.linspace(0.0, 1.0, 101)

fig, ax = plt.subplots(figsize=(8, 5))
for result in results:
    direction = result["values"] - truth_vector
    line_nll = np.asarray([
        float(vector_nll(jnp.asarray(truth_vector + alpha * direction), data_components))
        for alpha in alphas
    ])
    ax.plot(alphas, line_nll - float(truth_nll), label=result["label"])

ax.axhline(0.0, linestyle="--")
ax.set_xlabel(r"$\alpha$ along truth $\to$ fitted solution")
ax.set_ylabel("NLL - NLL(truth)")
ax.set_title("Likelihood paths from truth to fitted solutions")
ax.legend()
fig.tight_layout()
plt.show()


## 9. Immediate decision

Use the three-fit table and the two scans before changing `GenFit`.

A particularly strong failure signature is a Minuit result marked valid with a sizeable **positive** `DeltaNLL`: that fit has not found a minimum competitive with a point that was already known and available in the parameter space.
